# 第15章 硬件与虚拟机 - I/O系统与存储技术
# Chapter 15 Hardware & Virtual Machines - I/O Systems & Storage

---

**Cambridge A-Level Computer Science (9618) - A2 Level Paper 3**

本笔记本涵盖以下内容：
- I/O 处理方式：程序控制 I/O、中断驱动 I/O、DMA
- 存储层次结构 (Storage Hierarchy)
- RAID 级别 (0, 1, 5, 6, 10)
- SSD vs HDD 内部原理
- 云存储架构
- 交互式演示与可视化

In [ ]:
# ============================================================
# 环境配置 - 每个笔记本开头必须运行
# ============================================================
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets

# 中文字体配置
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['figure.dpi'] = 100

print("✅ 环境配置完成！")

---
## 1. I/O 处理方式

CPU 需要与外部设备（键盘、磁盘、网卡等）交换数据。这个过程称为 **I/O（Input/Output，输入/输出）**。

问题在于：外部设备的速度远慢于 CPU。CPU 不应该干等设备准备好数据。

有三种主要的 I/O 处理方式：

### 1.1 程序控制 I/O (Programmed I/O)

**工作方式**：
1. CPU 发送 I/O 请求给设备
2. CPU **反复检查**设备状态（轮询 Polling）："数据准备好了吗？"
3. 设备准备好后，CPU 读取数据
4. 重复直到所有数据传输完毕

**类比**：你站在微波炉前不停地看："好了没？好了没？好了没？"——完全浪费你的时间！

**优点**：
- 实现简单
- 不需要额外硬件

**缺点**：
- CPU **忙等待 (Busy Waiting)**：大量时间浪费在轮询上
- CPU 利用率极低
- CPU 不能做其他工作

### 1.2 中断驱动 I/O (Interrupt-driven I/O)

**工作方式**：
1. CPU 发送 I/O 请求给设备
2. CPU **继续执行其他任务**（不用等待！）
3. 设备准备好数据后，发送**中断信号 (Interrupt)** 给 CPU
4. CPU 暂停当前任务，执行**中断服务程序 (ISR, Interrupt Service Routine)** 来处理数据
5. ISR 完成后，CPU 恢复之前的任务

**类比**：你设了微波炉的定时器，然后去做别的事。微波炉"叮"一声（中断），你再回来取食物。

**优点**：
- CPU 不需要忙等待，利用率高
- 适合大多数 I/O 场景

**缺点**：
- 每传输一个字/字节就要中断一次 → 频繁中断有开销
- CPU 仍需参与每次数据传输
- 不适合大量数据传输（如读取整个文件）

### 1.3 直接内存访问 DMA (Direct Memory Access)

**工作方式**：
1. CPU 告诉 **DMA 控制器**："从磁盘的地址X读取N个字节到内存地址Y"
2. **DMA 控制器接管总线**，直接在设备和内存之间传输数据
3. CPU **完全不参与传输过程**，可以做其他事
4. DMA 传输完成后，DMA 控制器发送一次中断通知 CPU

**类比**：你雇了一个搬运工（DMA控制器），告诉他把仓库（磁盘）里的东西搬到客厅（内存），你就去做自己的事了。搬完了他通知你一声。

**优点**：
- CPU 几乎不参与传输，利用率最高
- 适合大量数据传输（磁盘读写、网络传输）
- 传输速度快（直接内存-设备传输，不经过 CPU）

**缺点**：
- 需要额外的 DMA 控制器硬件
- DMA 使用总线时，CPU 不能访问内存（总线竞争 Bus Contention）
- 实现更复杂

In [ ]:
# ============================================================
# 三种 I/O 方式对比可视化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
fig.suptitle('三种 I/O 处理方式对比', fontsize=16, fontweight='bold')

titles = [
    '程序控制 I/O\n(Programmed I/O)',
    '中断驱动 I/O\n(Interrupt-driven I/O)',
    'DMA\n(Direct Memory Access)'
]

# 时间线（10个时间单位）
time_steps = 10

# 程序控制I/O: CPU一直在轮询
cpu_programmed = ['轮询'] * 3 + ['传输'] + ['轮询'] * 3 + ['传输'] + ['轮询'] + ['完成']
cpu_colors_prog = ['#e74c3c'] * 3 + ['#3498db'] + ['#e74c3c'] * 3 + ['#3498db'] + ['#e74c3c'] + ['#27ae60']

# 中断驱动I/O: CPU做其他事，中断时传输
cpu_interrupt = ['其他'] * 3 + ['ISR'] + ['其他'] * 3 + ['ISR'] + ['其他'] + ['完成']
cpu_colors_int = ['#27ae60'] * 3 + ['#3498db'] + ['#27ae60'] * 3 + ['#3498db'] + ['#27ae60'] + ['#27ae60']

# DMA: CPU几乎全部做其他事
cpu_dma = ['设置'] + ['其他'] * 8 + ['完成']
cpu_colors_dma = ['#f39c12'] + ['#27ae60'] * 8 + ['#27ae60']

all_data = [
    (cpu_programmed, cpu_colors_prog),
    (cpu_interrupt, cpu_colors_int),
    (cpu_dma, cpu_colors_dma)
]

for ax, title, (labels, colors) in zip(axes, titles, all_data):
    ax.set_xlim(-0.5, 2)
    ax.set_ylim(-0.5, time_steps + 0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('时间 →', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks(range(time_steps))
    ax.set_yticklabels([f't{i}' for i in range(time_steps)])
    ax.invert_yaxis()
    
    for i, (label, color) in enumerate(zip(labels, colors)):
        ax.add_patch(FancyBboxPatch((0, i - 0.4), 1.5, 0.8, boxstyle='round,pad=0.05',
                                    facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.8))
        ax.text(0.75, i, label, ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# 图例
legend_patches = [
    mpatches.Patch(color='#e74c3c', label='CPU忙等待/轮询（浪费）'),
    mpatches.Patch(color='#3498db', label='CPU处理I/O数据'),
    mpatches.Patch(color='#27ae60', label='CPU做其他有用工作'),
    mpatches.Patch(color='#f39c12', label='CPU设置DMA'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=10)

plt.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()

# CPU 利用率计算
print("\n📊 CPU 利用率对比 (做有用工作的时间比例):")
useful_prog = sum(1 for l in cpu_programmed if l in ['传输', '完成'])
useful_int = sum(1 for l in cpu_interrupt if l in ['其他', '完成'])
useful_dma = sum(1 for l in cpu_dma if l in ['其他', '完成'])

print(f"   程序控制 I/O: {useful_prog}/{time_steps} = {useful_prog/time_steps:.0%}")
print(f"   中断驱动 I/O: {useful_int}/{time_steps} = {useful_int/time_steps:.0%}")
print(f"   DMA:          {useful_dma}/{time_steps} = {useful_dma/time_steps:.0%}")

In [ ]:
# ============================================================
# DMA 数据传输流程图
# ============================================================

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('DMA 直接内存访问工作流程', fontsize=16, fontweight='bold')

# CPU
ax.add_patch(FancyBboxPatch((0.5, 6), 3, 2, boxstyle='round,pad=0.2',
                            facecolor='#3498db', edgecolor='white', linewidth=2))
ax.text(2, 7, 'CPU', ha='center', va='center', fontsize=16, fontweight='bold', color='white')

# DMA Controller
ax.add_patch(FancyBboxPatch((5, 6), 3.5, 2, boxstyle='round,pad=0.2',
                            facecolor='#e74c3c', edgecolor='white', linewidth=2))
ax.text(6.75, 7, 'DMA\n控制器', ha='center', va='center', fontsize=14, fontweight='bold', color='white')

# 内存
ax.add_patch(FancyBboxPatch((10, 6), 3, 2, boxstyle='round,pad=0.2',
                            facecolor='#2ecc71', edgecolor='white', linewidth=2))
ax.text(11.5, 7, '内存\nRAM', ha='center', va='center', fontsize=14, fontweight='bold', color='white')

# I/O 设备
ax.add_patch(FancyBboxPatch((5, 1), 3.5, 2, boxstyle='round,pad=0.2',
                            facecolor='#f39c12', edgecolor='white', linewidth=2))
ax.text(6.75, 2, 'I/O 设备\n(磁盘)', ha='center', va='center', fontsize=14, fontweight='bold', color='white')

# 系统总线
ax.add_patch(FancyBboxPatch((0.5, 4.2), 12.5, 1, boxstyle='round,pad=0.1',
                            facecolor='#ecf0f1', edgecolor='#2c3e50', linewidth=2))
ax.text(6.75, 4.7, '系统总线 (System Bus)', ha='center', va='center', fontsize=12, fontweight='bold', color='#2c3e50')

# 连接线
ax.plot([2, 2], [6, 5.2], 'k-', linewidth=2)
ax.plot([6.75, 6.75], [6, 5.2], 'k-', linewidth=2)
ax.plot([11.5, 11.5], [6, 5.2], 'k-', linewidth=2)
ax.plot([6.75, 6.75], [4.2, 3], 'k-', linewidth=2)

# 步骤标注
steps = [
    (1.5, 8.5, '① CPU 发送DMA命令\n(源地址、目标地址、字节数)', '#3498db'),
    (6, 8.5, '② DMA控制器接管总线\n直接传输数据', '#e74c3c'),
    (11, 8.5, '③ 数据直接写入内存\n(不经过CPU)', '#2ecc71'),
    (11, 0.5, '④ 传输完成后\nDMA发送中断通知CPU', '#9b59b6'),
]

for x, y, text, color in steps:
    ax.text(x, y, text, fontsize=9, color=color, fontweight='bold',
           bbox=dict(facecolor='lightyellow', edgecolor=color, alpha=0.9, boxstyle='round,pad=0.3'))

# 数据流箭头（DMA直接传输）
ax.annotate('', xy=(10, 4.7), xytext=(8.5, 4.7),
            arrowprops=dict(arrowstyle='->', color='red', lw=3, linestyle='--'))
ax.text(9.25, 3.8, '数据流\n(不经CPU)', ha='center', fontsize=9, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

### 1.4 三种 I/O 方式对比总结

| 特性 | 程序控制 I/O | 中断驱动 I/O | DMA |
|------|-------------|-------------|-----|
| CPU 参与度 | 全程参与（忙等） | 部分参与（ISR） | 几乎不参与 |
| CPU 利用率 | 最低 | 中等 | 最高 |
| 传输速度 | 慢 | 中等 | 最快 |
| 额外硬件 | 无需 | 中断控制器 | DMA控制器 |
| 适用场景 | 简单、少量数据 | 一般 I/O | 大量数据传输 |
| 实现复杂度 | 简单 | 中等 | 复杂 |
| 每次传输单位 | 1字节/字 | 1字节/字 | 整个数据块 |

> **考试要点**：对比三种方式时，要从 CPU 利用率、传输效率、硬件需求、适用场景四个方面回答。

---
## 2. 存储层次结构 (Storage Hierarchy)

### 2.1 为什么需要层次结构？

**核心矛盾**：我们想要既快速又大容量的存储，但：
- 速度快的存储 → 价格贵、容量小（如寄存器、缓存）
- 容量大的存储 → 速度慢、价格便宜（如硬盘、磁带）

**解决方案**：多级存储层次结构，把最常用的数据放在最快的存储中。

### 2.2 存储层次（从快到慢）

| 层级 | 存储类型 | 典型容量 | 访问时间 | 每GB成本 |
|------|---------|---------|---------|----------|
| 1 | CPU 寄存器 (Register) | ~1 KB | ~0.3 ns | 极高 |
| 2 | L1 缓存 (Cache) | 32-64 KB | ~1 ns | 很高 |
| 3 | L2 缓存 | 256 KB - 1 MB | ~3-5 ns | 高 |
| 4 | L3 缓存 | 4-64 MB | ~10-20 ns | 较高 |
| 5 | 主存 RAM | 8-64 GB | ~50-100 ns | 中等 |
| 6 | SSD 固态硬盘 | 256 GB - 4 TB | ~0.1 ms | 较低 |
| 7 | HDD 机械硬盘 | 1-20 TB | ~5-10 ms | 低 |
| 8 | 磁带/光盘 (Tape) | 数百 TB | 秒-分钟 | 最低 |

In [ ]:
# ============================================================
# 存储层次金字塔可视化
# ============================================================

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(-8, 8)
ax.set_ylim(-1, 10)
ax.axis('off')
ax.set_title('存储层次结构 (Memory/Storage Hierarchy)', fontsize=16, fontweight='bold')

layers = [
    ('寄存器 (Registers)', '~1 KB', '0.3 ns', '#e74c3c', 1.2),
    ('L1 缓存 (Cache)', '32-64 KB', '1 ns', '#e67e22', 2.0),
    ('L2 缓存', '256 KB-1 MB', '3-5 ns', '#f39c12', 2.8),
    ('L3 缓存', '4-64 MB', '10-20 ns', '#f1c40f', 3.6),
    ('主存 RAM', '8-64 GB', '50-100 ns', '#2ecc71', 4.4),
    ('SSD 固态硬盘', '256 GB-4 TB', '0.1 ms', '#3498db', 5.2),
    ('HDD 机械硬盘', '1-20 TB', '5-10 ms', '#9b59b6', 6.0),
    ('磁带/云存储', '数百 TB+', '秒-分钟', '#95a5a6', 6.8),
]

for i, (name, capacity, speed, color, width) in enumerate(layers):
    y = 8.5 - i * 1.1
    half_w = width
    # 梯形
    trap = plt.Polygon([
        [-half_w, y], [half_w, y],
        [half_w + 0.4, y - 0.9], [-half_w - 0.4, y - 0.9]
    ], facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(trap)
    
    ax.text(0, y - 0.45, name, ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    
    # 左侧：容量
    ax.text(-half_w - 1.5, y - 0.45, capacity, ha='right', va='center', fontsize=9, color=color, fontweight='bold')
    
    # 右侧：速度
    ax.text(half_w + 1.5, y - 0.45, speed, ha='left', va='center', fontsize=9, color=color, fontweight='bold')

# 标签
ax.annotate('', xy=(-7.5, 0), xytext=(-7.5, 9),
            arrowprops=dict(arrowstyle='->', lw=2, color='#e74c3c'))
ax.text(-7.8, 4.5, '容量增大\n成本降低', ha='center', va='center', fontsize=10, color='#e74c3c',
        fontweight='bold', rotation=90)

ax.annotate('', xy=(7.5, 9), xytext=(7.5, 0),
            arrowprops=dict(arrowstyle='->', lw=2, color='#27ae60'))
ax.text(7.8, 4.5, '速度增快\n成本增高', ha='center', va='center', fontsize=10, color='#27ae60',
        fontweight='bold', rotation=90)

ax.text(-5, 9.3, '← 容量', fontsize=11, fontweight='bold', color='#2c3e50')
ax.text(4, 9.3, '速度 →', fontsize=11, fontweight='bold', color='#2c3e50')

# 分类标注
ax.add_patch(FancyBboxPatch((-7.5, 5.5), 1.5, 3.5, boxstyle='round,pad=0.1',
                            facecolor='none', edgecolor='#e74c3c', linewidth=1.5, linestyle='--'))
ax.text(-6.75, 5.2, '易失性\n(Volatile)', ha='center', fontsize=8, color='#e74c3c')

ax.add_patch(FancyBboxPatch((-7.5, -0.5), 1.5, 5.5, boxstyle='round,pad=0.1',
                            facecolor='none', edgecolor='#3498db', linewidth=1.5, linestyle='--'))
ax.text(-6.75, -0.8, '非易失性\n(Non-volatile)', ha='center', fontsize=8, color='#3498db')

plt.tight_layout()
plt.show()

### 2.3 缓存的工作原理

**缓存 (Cache)** 是CPU和主存之间的高速缓冲存储器。

**工作原理**：
1. CPU 需要数据时，先查缓存
2. **缓存命中 (Cache Hit)**：数据在缓存中 → 直接读取（快！）
3. **缓存未命中 (Cache Miss)**：数据不在缓存中 → 从主存读取，同时存入缓存

**命中率 (Hit Rate)** = 缓存命中次数 / 总访问次数

**局部性原理 (Locality Principle)** —— 缓存有效的原因：
- **时间局部性 (Temporal Locality)**：最近访问的数据很可能再次被访问
- **空间局部性 (Spatial Locality)**：被访问地址附近的数据很可能也会被访问

In [ ]:
# ============================================================
# 缓存命中率对有效访问时间的影响
# ============================================================

def cache_performance(hit_rate_pct=90, cache_time_ns=2, ram_time_ns=100):
    """计算并可视化缓存性能"""
    hit_rate = hit_rate_pct / 100
    
    # 有效访问时间 = hit_rate * cache_time + (1 - hit_rate) * ram_time
    effective_time = hit_rate * cache_time_ns + (1 - hit_rate) * ram_time_ns
    
    # 不同命中率下的有效访问时间
    hit_rates = np.linspace(0, 1, 100)
    eff_times = hit_rates * cache_time_ns + (1 - hit_rates) * ram_time_ns
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左图：有效访问时间曲线
    ax1.plot(hit_rates * 100, eff_times, 'b-', linewidth=2)
    ax1.plot(hit_rate_pct, effective_time, 'ro', markersize=10, zorder=5)
    ax1.axhline(y=cache_time_ns, color='green', linestyle='--', alpha=0.5, label=f'缓存时间 = {cache_time_ns} ns')
    ax1.axhline(y=ram_time_ns, color='red', linestyle='--', alpha=0.5, label=f'RAM时间 = {ram_time_ns} ns')
    ax1.set_xlabel('命中率 (%)', fontsize=12)
    ax1.set_ylabel('有效访问时间 (ns)', fontsize=12)
    ax1.set_title('命中率 vs 有效访问时间', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # 标注当前点
    ax1.annotate(f'{effective_time:.1f} ns', xy=(hit_rate_pct, effective_time),
                xytext=(hit_rate_pct - 15, effective_time + 15),
                fontsize=11, fontweight='bold', color='red',
                arrowprops=dict(arrowstyle='->', color='red'))
    
    # 右图：加速比
    speedups = ram_time_ns / eff_times
    current_speedup = ram_time_ns / effective_time
    ax2.plot(hit_rates * 100, speedups, 'g-', linewidth=2)
    ax2.plot(hit_rate_pct, current_speedup, 'ro', markersize=10, zorder=5)
    ax2.set_xlabel('命中率 (%)', fontsize=12)
    ax2.set_ylabel('加速比 (相对于无缓存)', fontsize=12)
    ax2.set_title('缓存带来的加速比', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.annotate(f'{current_speedup:.1f}x', xy=(hit_rate_pct, current_speedup),
                xytext=(hit_rate_pct - 15, current_speedup + 2),
                fontsize=11, fontweight='bold', color='red',
                arrowprops=dict(arrowstyle='->', color='red'))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 计算结果:")
    print(f"   缓存命中率: {hit_rate_pct}%")
    print(f"   有效访问时间 = {hit_rate:.2f} × {cache_time_ns}ns + {1-hit_rate:.2f} × {ram_time_ns}ns")
    print(f"                = {hit_rate*cache_time_ns:.1f} + {(1-hit_rate)*ram_time_ns:.1f}")
    print(f"                = {effective_time:.1f} ns")
    print(f"   加速比 = {ram_time_ns}/{effective_time:.1f} = {current_speedup:.2f}x")

widgets.interact(
    cache_performance,
    hit_rate_pct=widgets.IntSlider(min=0, max=100, value=90, description='命中率%:'),
    cache_time_ns=widgets.IntSlider(min=1, max=20, value=2, description='缓存(ns):'),
    ram_time_ns=widgets.IntSlider(min=20, max=200, value=100, description='RAM(ns):')
)

---
## 3. RAID 技术

### 3.1 什么是 RAID？

**RAID (Redundant Array of Independent Disks)** = 独立磁盘冗余阵列

**核心思想**：将多个物理磁盘组合成一个逻辑磁盘，以提供：
- **更高性能**：数据分布在多个磁盘上，可以并行读写
- **数据冗余**：即使一个磁盘损坏，数据不会丢失

### 3.2 RAID 级别

#### RAID 0 - 条带化 (Striping)
- 数据**均匀分布**在所有磁盘上（条带化）
- **无冗余**：任何一个磁盘损坏 → 所有数据丢失！
- 读写速度最快（N倍并行）
- 可用容量 = 所有磁盘之和
- 用途：追求速度不在乎安全（如视频编辑暂存）

#### RAID 1 - 镜像 (Mirroring)
- 数据**完全复制**到另一个磁盘（镜像）
- 最高冗余：一个磁盘坏了，另一个还有完整数据
- 读速度提升（可从两个磁盘读），写速度不变
- 可用容量 = 总容量的 50%（一半用于镜像）
- 用途：重要数据保护

#### RAID 5 - 带分布式奇偶校验的条带化 (Striping with Distributed Parity)
- 数据条带化分布，**奇偶校验 (Parity)** 也分布在各磁盘上
- 最少需要 3 个磁盘
- 可容忍 **1 个磁盘** 故障
- 可用容量 = (N-1) × 单盘容量（一个盘的容量用于奇偶校验）
- 兼顾性能和冗余
- 用途：最常用的企业级 RAID

#### RAID 6 - 双重奇偶校验 (Double Parity)
- 类似 RAID 5，但有**两组**奇偶校验
- 最少需要 4 个磁盘
- 可容忍 **2 个磁盘** 同时故障
- 可用容量 = (N-2) × 单盘容量
- 用途：对数据安全要求更高的场景

#### RAID 10 (1+0) - 镜像 + 条带化
- 先镜像 (RAID 1)，再条带化 (RAID 0)
- 最少需要 4 个磁盘
- 兼具 RAID 0 的速度和 RAID 1 的冗余
- 可用容量 = 总容量的 50%
- 用途：数据库服务器（高性能 + 高可靠）

In [ ]:
# ============================================================
# RAID 级别可视化
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('RAID 级别可视化对比', fontsize=16, fontweight='bold')

def draw_raid(ax, title, disks_data, n_disks, subtitle=''):
    """绘制 RAID 配置图
    disks_data: list of lists, 每个子列表是一个磁盘的数据块
    """
    ax.set_xlim(-0.5, n_disks * 2)
    ax.set_ylim(-1, 6)
    ax.axis('off')
    ax.set_title(f'{title}\n{subtitle}', fontsize=12, fontweight='bold')
    
    color_map = {
        'A': '#3498db', 'B': '#e74c3c', 'C': '#2ecc71', 'D': '#f39c12',
        'E': '#9b59b6', 'F': '#1abc9c', 'G': '#e67e22', 'H': '#34495e',
        'P': '#95a5a6', 'Q': '#7f8c8d',  # Parity
        'A1': '#3498db', 'A2': '#3498db', 'B1': '#e74c3c', 'B2': '#e74c3c',
        'C1': '#2ecc71', 'C2': '#2ecc71', 'D1': '#f39c12', 'D2': '#f39c12',
    }
    
    for d, disk in enumerate(disks_data):
        x = d * 1.8 + 0.3
        # 磁盘外框
        ax.add_patch(FancyBboxPatch((x-0.1, 0.3), 1.4, 4.5, boxstyle='round,pad=0.1',
                                    facecolor='#ecf0f1', edgecolor='#2c3e50', linewidth=1.5))
        ax.text(x + 0.6, 5, f'磁盘{d+1}', ha='center', va='center', fontsize=9, fontweight='bold')
        
        for b, block in enumerate(disk):
            y = 3.8 - b * 1.0
            is_parity = block.startswith('P') or block.startswith('Q')
            is_mirror = block.endswith("'")  # 镜像副本
            
            base_block = block.rstrip("'")
            color = color_map.get(base_block, '#bdc3c7')
            if is_parity:
                color = '#95a5a6'
            alpha = 0.5 if is_mirror else 0.8
            
            ax.add_patch(FancyBboxPatch((x, y), 1.2, 0.8, boxstyle='round,pad=0.05',
                                        facecolor=color, edgecolor='white', linewidth=1, alpha=alpha))
            ax.text(x + 0.6, y + 0.4, block, ha='center', va='center',
                   fontsize=9, fontweight='bold', color='white')

# RAID 0
draw_raid(axes[0,0], 'RAID 0 - 条带化', 
          [['A', 'C', 'E', 'G'], ['B', 'D', 'F', 'H']],
          2, '无冗余，最快速度')

# RAID 1
draw_raid(axes[0,1], 'RAID 1 - 镜像',
          [['A', 'B', 'C', 'D'], ["A'", "B'", "C'", "D'"]],
          2, '完全镜像，50%容量')

# RAID 5
draw_raid(axes[0,2], 'RAID 5 - 分布式奇偶',
          [['A', 'D', 'P3'], ['B', 'P2', 'G'], ['P1', 'E', 'H']],
          3, '1个校验盘容量，容忍1盘故障')

# RAID 6
draw_raid(axes[1,0], 'RAID 6 - 双重奇偶',
          [['A', 'P2', 'Q3'], ['B', 'Q2', 'F'], ['P1', 'D', 'G'], ['Q1', 'E', 'H']],
          4, '2个校验盘容量，容忍2盘故障')

# RAID 10
draw_raid(axes[1,1], 'RAID 10 - 镜像+条带化',
          [['A', 'C'], ["A'", "C'"], ['B', 'D'], ["B'", "D'"]],
          4, '50%容量，高速+高可靠')

# 对比表
ax = axes[1,2]
ax.axis('off')
ax.set_title('RAID 级别对比总结', fontsize=12, fontweight='bold')

table_data = [
    ['级别', '最少盘数', '可用容量', '容错'],
    ['RAID 0', '2', 'N盘', '0盘'],
    ['RAID 1', '2', 'N/2盘', '1盘'],
    ['RAID 5', '3', '(N-1)盘', '1盘'],
    ['RAID 6', '4', '(N-2)盘', '2盘'],
    ['RAID 10', '4', 'N/2盘', '每组1盘'],
]

for i, row in enumerate(table_data):
    y = 4.5 - i * 0.8
    bg = '#2c3e50' if i == 0 else ('#f8f9fa' if i % 2 == 0 else '#ffffff')
    text_color = 'white' if i == 0 else '#2c3e50'
    for j, cell in enumerate(row):
        x = j * 2.2 - 0.5
        ax.add_patch(plt.Rectangle((x, y-0.3), 2.1, 0.7, facecolor=bg, edgecolor='#dee2e6'))
        ax.text(x + 1.05, y+0.05, cell, ha='center', va='center', fontsize=9,
               fontweight='bold' if i == 0 else 'normal', color=text_color)

ax.set_xlim(-1, 9)
ax.set_ylim(-0.5, 5.5)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 交互式 RAID 容量/性能计算器
# ============================================================

def raid_calculator(n_disks=4, disk_size_tb=2, raid_level='RAID 5'):
    """计算 RAID 配置的可用容量和特性"""
    
    configs = {
        'RAID 0': {
            'min_disks': 2,
            'usable': lambda n, s: n * s,
            'fault_tolerance': 0,
            'read_speed': lambda n: f'{n}x',
            'write_speed': lambda n: f'{n}x',
        },
        'RAID 1': {
            'min_disks': 2,
            'usable': lambda n, s: (n // 2) * s,
            'fault_tolerance': 1,
            'read_speed': lambda n: f'{n}x',
            'write_speed': lambda n: '1x',
        },
        'RAID 5': {
            'min_disks': 3,
            'usable': lambda n, s: (n - 1) * s,
            'fault_tolerance': 1,
            'read_speed': lambda n: f'{n-1}x',
            'write_speed': lambda n: f'{(n-1)//2}x',
        },
        'RAID 6': {
            'min_disks': 4,
            'usable': lambda n, s: (n - 2) * s,
            'fault_tolerance': 2,
            'read_speed': lambda n: f'{n-2}x',
            'write_speed': lambda n: f'{(n-2)//2}x',
        },
        'RAID 10': {
            'min_disks': 4,
            'usable': lambda n, s: (n // 2) * s,
            'fault_tolerance': 1,  # per mirror group
            'read_speed': lambda n: f'{n}x',
            'write_speed': lambda n: f'{n//2}x',
        },
    }
    
    config = configs[raid_level]
    
    if n_disks < config['min_disks']:
        print(f"❌ {raid_level} 至少需要 {config['min_disks']} 个磁盘！当前只有 {n_disks} 个。")
        return
    
    if raid_level in ['RAID 1', 'RAID 10'] and n_disks % 2 != 0:
        print(f"❌ {raid_level} 需要偶数个磁盘！")
        return
    
    total = n_disks * disk_size_tb
    usable = config['usable'](n_disks, disk_size_tb)
    overhead = total - usable
    efficiency = usable / total * 100
    
    # 可视化
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # 左图：容量分配
    ax1.barh(['总容量', '可用容量', '冗余开销'], [total, usable, overhead],
            color=['#3498db', '#27ae60', '#e74c3c'])
    ax1.set_xlabel('容量 (TB)', fontsize=11)
    ax1.set_title(f'{raid_level} 容量分配', fontsize=12, fontweight='bold')
    for i, v in enumerate([total, usable, overhead]):
        ax1.text(v + 0.1, i, f'{v:.1f} TB', va='center', fontsize=10, fontweight='bold')
    
    # 右图：各 RAID 级别效率对比
    raid_names = list(configs.keys())
    efficiencies = []
    for name in raid_names:
        c = configs[name]
        if n_disks >= c['min_disks']:
            if name in ['RAID 1', 'RAID 10'] and n_disks % 2 != 0:
                efficiencies.append(0)
            else:
                eff = c['usable'](n_disks, disk_size_tb) / total * 100
                efficiencies.append(eff)
        else:
            efficiencies.append(0)
    
    colors = ['#e74c3c' if name == raid_level else '#3498db' for name in raid_names]
    bars = ax2.bar(raid_names, efficiencies, color=colors)
    ax2.set_ylabel('空间利用率 (%)', fontsize=11)
    ax2.set_title(f'{n_disks} 个磁盘下各 RAID 利用率对比', fontsize=12, fontweight='bold')
    ax2.set_ylim(0, 110)
    for bar, eff in zip(bars, efficiencies):
        if eff > 0:
            ax2.text(bar.get_x() + bar.get_width()/2, eff + 2, f'{eff:.0f}%',
                    ha='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 {raid_level} 配置详情:")
    print(f"   磁盘数量: {n_disks} × {disk_size_tb} TB")
    print(f"   总原始容量: {total:.1f} TB")
    print(f"   可用容量: {usable:.1f} TB")
    print(f"   冗余开销: {overhead:.1f} TB")
    print(f"   空间利用率: {efficiency:.0f}%")
    print(f"   最大容错: {config['fault_tolerance']} 个磁盘同时故障")
    print(f"   读取速度: {config['read_speed'](n_disks)}")
    print(f"   写入速度: {config['write_speed'](n_disks)}")

widgets.interact(
    raid_calculator,
    n_disks=widgets.IntSlider(min=2, max=12, value=4, description='磁盘数:'),
    disk_size_tb=widgets.FloatSlider(min=0.5, max=20, value=2, step=0.5, description='单盘TB:'),
    raid_level=widgets.Dropdown(options=['RAID 0', 'RAID 1', 'RAID 5', 'RAID 6', 'RAID 10'],
                               value='RAID 5', description='RAID级别:')
)

---
## 4. SSD vs HDD 内部原理

### 4.1 HDD (Hard Disk Drive) 机械硬盘

**工作原理**：
- 数据存储在高速旋转的**磁性盘片 (Platter)** 上
- **读写磁头 (Read/Write Head)** 在盘片上方移动，读取/写入数据
- 数据按**磁道 (Track)** 和**扇区 (Sector)** 组织

**访问时间组成**：
- **寻道时间 (Seek Time)**：磁头移动到正确磁道的时间（~3-15 ms）
- **旋转延迟 (Rotational Latency)**：等待目标扇区转到磁头下方（~2-8 ms）
- **传输时间 (Transfer Time)**：实际读写数据的时间（通常很短）

**优点**：容量大、价格低、寿命长（无写入次数限制）
**缺点**：速度慢、有机械部件（怕震动）、噪音、功耗高

### 4.2 SSD (Solid State Drive) 固态硬盘

**工作原理**：
- 使用 **NAND Flash 闪存**芯片存储数据
- 没有机械部件，完全电子化
- 数据以**页 (Page)** 为单位读写（通常 4-16 KB）
- 数据以**块 (Block)** 为单位擦除（通常 256-512 KB）

**关键技术**：
- **磨损均衡 (Wear Leveling)**：NAND 闪存每个单元有写入次数限制（~3000-100000次），控制器确保写入均匀分布，延长寿命
- **TRIM 命令**：告诉 SSD 哪些数据已删除，可以预先擦除，提高写入性能
- **垃圾回收 (Garbage Collection)**：整理碎片数据，释放可用块

**NAND 类型**：
| 类型 | 每单元位数 | 速度 | 寿命 | 价格 |
|------|-----------|------|------|------|
| SLC | 1 bit | 最快 | 最长 | 最贵 |
| MLC | 2 bits | 快 | 中 | 中 |
| TLC | 3 bits | 中 | 较短 | 便宜 |
| QLC | 4 bits | 慢 | 最短 | 最便宜 |

In [ ]:
# ============================================================
# SSD vs HDD 性能对比可视化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('SSD vs HDD 性能对比', fontsize=16, fontweight='bold')

# 数据
categories = ['顺序读取\n(MB/s)', '顺序写入\n(MB/s)', '随机读取\n(IOPS)', '随机写入\n(IOPS)']

# 图1: 顺序读写速度
ax = axes[0]
metrics = ['顺序读取', '顺序写入']
hdd_speed = [150, 120]  # MB/s
sata_ssd = [550, 520]   # MB/s
nvme_ssd = [3500, 3000]  # MB/s

x = np.arange(len(metrics))
width = 0.25
ax.bar(x - width, hdd_speed, width, label='HDD (7200 RPM)', color='#95a5a6')
ax.bar(x, sata_ssd, width, label='SATA SSD', color='#3498db')
ax.bar(x + width, nvme_ssd, width, label='NVMe SSD', color='#e74c3c')
ax.set_ylabel('速度 (MB/s)', fontsize=11)
ax.set_title('顺序读写速度', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# 图2: 随机 IOPS
ax = axes[1]
metrics2 = ['随机读取', '随机写入']
hdd_iops = [100, 80]
sata_iops = [90000, 80000]
nvme_iops = [500000, 400000]

x = np.arange(len(metrics2))
ax.bar(x - width, hdd_iops, width, label='HDD', color='#95a5a6')
ax.bar(x, sata_iops, width, label='SATA SSD', color='#3498db')
ax.bar(x + width, nvme_iops, width, label='NVMe SSD', color='#e74c3c')
ax.set_ylabel('IOPS (每秒I/O操作数)', fontsize=11)
ax.set_title('随机读写 IOPS', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics2)
ax.legend(fontsize=9)
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

# 图3: 其他特性雷达图模拟（使用条形图）
ax = axes[2]
features = ['速度', '容量/$', '耐用性', '抗震', '静音', '功耗']
hdd_scores = [2, 9, 7, 2, 3, 3]
ssd_scores = [9, 5, 6, 9, 10, 8]

y = np.arange(len(features))
ax.barh(y - 0.15, hdd_scores, 0.3, label='HDD', color='#95a5a6')
ax.barh(y + 0.15, ssd_scores, 0.3, label='SSD', color='#e74c3c')
ax.set_xlabel('评分 (1-10)', fontsize=11)
ax.set_title('综合特性对比', fontsize=12, fontweight='bold')
ax.set_yticks(y)
ax.set_yticklabels(features)
ax.legend(fontsize=9)
ax.set_xlim(0, 11)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("💡 关键对比总结:")
print("   - NVMe SSD 顺序读取速度是 HDD 的 ~23 倍")
print("   - NVMe SSD 随机读取 IOPS 是 HDD 的 ~5000 倍！")
print("   - HDD 在 $/GB 和总容量方面仍有优势")

In [ ]:
# ============================================================
# SSD 磨损均衡 (Wear Leveling) 演示
# ============================================================

np.random.seed(42)

n_blocks = 16
n_writes = 200

# 无磨损均衡：写入集中在几个块
no_wl = np.zeros(n_blocks)
hot_blocks = [0, 1, 2, 3]  # 热点块
for _ in range(n_writes):
    if np.random.random() < 0.8:  # 80% 写入热点块
        block = np.random.choice(hot_blocks)
    else:
        block = np.random.randint(0, n_blocks)
    no_wl[block] += 1

# 有磨损均衡：写入均匀分布
with_wl = np.zeros(n_blocks)
for _ in range(n_writes):
    # 选择写入次数最少的块
    min_block = np.argmin(with_wl)
    with_wl[min_block] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 无磨损均衡
colors1 = ['#e74c3c' if w > n_writes/n_blocks * 2 else '#3498db' for w in no_wl]
ax1.bar(range(n_blocks), no_wl, color=colors1, edgecolor='white')
ax1.axhline(y=n_writes/n_blocks, color='green', linestyle='--', label=f'理想均值 = {n_writes/n_blocks:.0f}')
ax1.set_xlabel('闪存块编号', fontsize=11)
ax1.set_ylabel('写入次数', fontsize=11)
ax1.set_title('无磨损均衡 (No Wear Leveling)\n某些块快速磨损！', fontsize=12, fontweight='bold', color='red')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# 有磨损均衡
ax2.bar(range(n_blocks), with_wl, color='#27ae60', edgecolor='white')
ax2.axhline(y=n_writes/n_blocks, color='green', linestyle='--', label=f'理想均值 = {n_writes/n_blocks:.0f}')
ax2.set_xlabel('闪存块编号', fontsize=11)
ax2.set_ylabel('写入次数', fontsize=11)
ax2.set_title('有磨损均衡 (Wear Leveling)\n写入均匀分布', fontsize=12, fontweight='bold', color='green')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, max(no_wl) * 1.1)  # 用相同的 y 轴范围

plt.tight_layout()
plt.show()

print(f"\n📊 磨损均衡效果:")
print(f"   无均衡 - 最大写入次数: {int(max(no_wl))}, 最小: {int(min(no_wl))}, 差异: {int(max(no_wl)-min(no_wl))}")
print(f"   有均衡 - 最大写入次数: {int(max(with_wl))}, 最小: {int(min(with_wl))}, 差异: {int(max(with_wl)-min(with_wl))}")
print(f"\n💡 磨损均衡使所有块的写入次数接近均值，")
print(f"   大幅延长了 SSD 的使用寿命！")

---
## 5. 云存储架构

### 5.1 什么是云存储？

**云存储 (Cloud Storage)** 是一种将数据存储在远程服务器上，通过互联网访问的存储服务。

用户不需要了解底层硬件细节，只需要通过网络上传和下载文件。

### 5.2 云存储的关键技术

| 技术 | 说明 |
|------|------|
| **分布式存储** | 数据分散存储在多个服务器/数据中心，避免单点故障 |
| **数据冗余** | 每份数据至少复制 3 份（通常分布在不同地理位置） |
| **虚拟化** | 使用虚拟机技术动态分配存储资源 |
| **负载均衡** | 将用户请求分配到不同服务器，避免过载 |
| **加密** | 数据在传输和存储时加密保护 |
| **CDN** | 内容分发网络，将数据缓存到离用户最近的节点 |

### 5.3 云存储的优势

- **可扩展性 (Scalability)**：按需增加或减少存储空间
- **随时随地访问**：只要有网络就能访问数据
- **成本效益**：无需购买和维护本地存储设备（按使用量付费）
- **自动备份**：云服务商自动维护数据冗余
- **协作方便**：多人可以同时访问和编辑文件

### 5.4 云存储的劣势

- **依赖网络**：没有网络就无法访问
- **隐私担忧**：数据存储在第三方服务器
- **延迟**：访问远程数据比本地存储慢
- **持续成本**：长期使用费用可能超过购买本地存储
- **安全风险**：潜在的数据泄露（尽管有加密）

In [ ]:
# ============================================================
# 云存储架构示意图
# ============================================================

fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 15)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('云存储架构 (Cloud Storage Architecture)', fontsize=16, fontweight='bold')

# 用户设备
devices = [('手机', 1, 8.5), ('笔记本', 3.5, 8.5), ('平板', 6, 8.5)]
for name, x, y in devices:
    ax.add_patch(FancyBboxPatch((x, y), 1.8, 1, boxstyle='round,pad=0.1',
                                facecolor='#3498db', edgecolor='white', linewidth=2))
    ax.text(x+0.9, y+0.5, name, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# 互联网
ax.add_patch(FancyBboxPatch((1, 6.5), 8, 1.2, boxstyle='round,pad=0.2',
                            facecolor='#f39c12', edgecolor='white', linewidth=2, alpha=0.3))
ax.text(5, 7.1, '互联网 (Internet) - 加密传输', ha='center', va='center',
        fontsize=12, fontweight='bold', color='#e67e22')

# 连接线
for name, x, y in devices:
    ax.plot([x+0.9, x+0.9], [y, 7.7], 'k--', linewidth=1, alpha=0.5)

# 负载均衡器
ax.add_patch(FancyBboxPatch((3, 4.8), 4, 1.2, boxstyle='round,pad=0.15',
                            facecolor='#9b59b6', edgecolor='white', linewidth=2))
ax.text(5, 5.4, '负载均衡器\nLoad Balancer', ha='center', va='center',
        fontsize=11, fontweight='bold', color='white')
ax.plot([5, 5], [6.5, 6.0], 'k-', linewidth=2)

# 数据中心
dc_positions = [(0.5, 1), (5, 1), (9.5, 1)]
dc_names = ['数据中心 A\n(北京)', '数据中心 B\n(上海)', '数据中心 C\n(广州)']
dc_colors = ['#27ae60', '#3498db', '#e74c3c']

for (x, y), name, color in zip(dc_positions, dc_names, dc_colors):
    ax.add_patch(FancyBboxPatch((x, y), 4.5, 3, boxstyle='round,pad=0.15',
                                facecolor=color, edgecolor='white', linewidth=2, alpha=0.2))
    ax.text(x+2.25, y+2.7, name, ha='center', va='center', fontsize=9, fontweight='bold', color=color)
    
    # 服务器
    for i in range(3):
        sx = x + 0.3 + i * 1.5
        ax.add_patch(FancyBboxPatch((sx, y+0.3), 1.1, 1.8, boxstyle='round,pad=0.05',
                                    facecolor=color, edgecolor='white', alpha=0.6))
        ax.text(sx+0.55, y+1.2, f'S{i+1}', ha='center', va='center',
               fontsize=8, fontweight='bold', color='white')
    
    # 连接线
    ax.plot([x+2.25, 5], [y+3, 4.8], 'k--', linewidth=1, alpha=0.5)

# 说明
ax.text(12, 8, '特点:\n1. 数据冗余(3副本)\n2. 地理分布\n3. 加密传输\n4. 负载均衡\n5. 自动故障转移',
        fontsize=9, verticalalignment='top',
        bbox=dict(facecolor='lightyellow', edgecolor='gray', alpha=0.9, boxstyle='round,pad=0.5'))

plt.tight_layout()
plt.show()

---
## 6. 存储容量计算

### 考试常见计算题类型

In [ ]:
# ============================================================
# 交互式存储容量计算器
# ============================================================

def storage_calculator(calc_type='磁盘容量'):
    """多种存储计算"""
    
    if calc_type == '磁盘容量':
        print("📐 磁盘容量计算公式:")
        print("   容量 = 磁头数 × 磁道数 × 扇区数 × 扇区大小")
        print("="*50)
        
        heads = 4       # 磁头数（= 盘面数）
        tracks = 10000  # 每面磁道数
        sectors = 63    # 每磁道扇区数
        sector_size = 512  # 每扇区字节数
        
        capacity_bytes = heads * tracks * sectors * sector_size
        capacity_gb = capacity_bytes / (1024**3)
        
        print(f"\n例题：一个硬盘有:")
        print(f"   {heads} 个磁头（盘面）")
        print(f"   每面 {tracks:,} 条磁道")
        print(f"   每磁道 {sectors} 个扇区")
        print(f"   每扇区 {sector_size} 字节")
        print(f"\n计算:")
        print(f"   容量 = {heads} × {tracks:,} × {sectors} × {sector_size}")
        print(f"        = {capacity_bytes:,} 字节")
        print(f"        = {capacity_bytes/(1024**2):,.1f} MB")
        print(f"        ≈ {capacity_gb:.2f} GB")
    
    elif calc_type == 'RAID容量':
        print("📐 RAID 可用容量计算:")
        print("="*50)
        
        disk_size = 2  # TB
        n_disks = 6
        
        print(f"\n已知: {n_disks} 个 {disk_size}TB 磁盘")
        print(f"总原始容量 = {n_disks} × {disk_size} = {n_disks * disk_size} TB")
        print()
        
        raids = {
            'RAID 0': n_disks * disk_size,
            'RAID 1': (n_disks // 2) * disk_size,
            'RAID 5': (n_disks - 1) * disk_size,
            'RAID 6': (n_disks - 2) * disk_size,
            'RAID 10': (n_disks // 2) * disk_size,
        }
        
        for raid, usable in raids.items():
            eff = usable / (n_disks * disk_size) * 100
            print(f"   {raid:>8}: 可用 {usable:>5.1f} TB (效率 {eff:>5.1f}%)")
    
    elif calc_type == '传输时间':
        print("📐 文件传输时间计算:")
        print("   时间 = 文件大小 / 传输速率")
        print("="*50)
        
        file_size_gb = 50  # GB
        file_size_bits = file_size_gb * 8 * (1024**3)  # 转为 bits
        
        speeds = {
            'HDD (SATA)': 150 * 8 * (1024**2),   # 150 MB/s → bits/s
            'SATA SSD': 550 * 8 * (1024**2),
            'NVMe SSD': 3500 * 8 * (1024**2),
            'USB 3.0': 625 * 8 * (1024**2),
            '千兆网络': 1 * (1000**3),  # 1 Gbps
        }
        
        print(f"\n例题: 传输 {file_size_gb} GB 文件需要多长时间？")
        print()
        for name, speed_bps in speeds.items():
            time_s = (file_size_gb * 1024 * 1024 * 1024 * 8) / speed_bps
            if time_s < 60:
                time_str = f"{time_s:.1f} 秒"
            else:
                time_str = f"{time_s/60:.1f} 分钟"
            print(f"   {name:>12}: {time_str}")

widgets.interact(
    storage_calculator,
    calc_type=widgets.Dropdown(
        options=['磁盘容量', 'RAID容量', '传输时间'],
        value='磁盘容量',
        description='计算类型:'
    )
)

---
## 7. 综合对比与总结

### I/O 处理方式考试要点

| 考题方向 | 关键答案 |
|---------|----------|
| 为什么 DMA 比 Programmed I/O 好？ | DMA 不需要 CPU 参与数据传输，CPU 可以执行其他任务，利用率更高 |
| DMA 如何工作？ | CPU 设置 DMA 参数 → DMA 控制器接管总线 → 直接在设备和内存间传输 → 完成后发中断 |
| 中断驱动的优势？ | CPU 不需要忙等待，可在等待时执行其他任务 |

### 存储层次考试要点

| 考题方向 | 关键答案 |
|---------|----------|
| 为什么需要缓存？ | CPU 和 RAM 速度差距大，缓存弥补这个差距（局部性原理） |
| 缓存命中/未命中？ | Hit: 数据在缓存中，直接读取。Miss: 不在缓存，从 RAM 读并存入缓存 |
| SSD vs HDD？ | SSD 更快、无机械部件、抗震；HDD 更便宜、容量更大 |

### RAID 考试要点

| 考题方向 | 关键答案 |
|---------|----------|
| RAID 0 vs RAID 1？ | RAID 0 只有条带化（快但无冗余），RAID 1 镜像（安全但50%浪费）|
| RAID 5 的工作原理？ | 条带化 + 分布式奇偶校验，容忍 1 盘故障，(N-1) 盘可用 |
| 计算 RAID 可用容量？ | 套用公式即可 |

---
## 8. 练习题 (Practice Exercises)

### 练习 1：I/O 方式对比

**题目**：比较程序控制 I/O、中断驱动 I/O 和 DMA 三种方式。对于每种方式，说明 CPU 的参与程度和适用场景。(6分)

<details>
<summary>点击查看答案</summary>

**程序控制 I/O**：CPU 全程参与，反复轮询设备状态（忙等待）。CPU 利用率最低。适用于简单设备和少量数据传输。(2分)

**中断驱动 I/O**：CPU 发出请求后继续执行其他任务，设备就绪后发送中断信号，CPU 暂停执行 ISR 处理数据。CPU 利用率较高。适用于一般 I/O 操作。(2分)

**DMA**：CPU 只需设置 DMA 参数，DMA 控制器直接在设备和内存间传输数据，完成后发一次中断。CPU 几乎不参与。适用于大量数据传输（如磁盘读写）。(2分)
</details>

### 练习 2：存储层次

**题目**：

(a) 解释什么是缓存 (cache) 以及它为什么能提高性能。(3分)

(b) 某系统缓存访问时间为 5ns，主存访问时间为 80ns，缓存命中率为 95%。计算平均有效访问时间。(3分)

<details>
<summary>点击查看答案</summary>

(a) 缓存是位于 CPU 和主存之间的高速存储器。(1分) 它利用局部性原理（最近使用的数据很可能再次使用），将常用数据存储在快速的缓存中。(1分) 当 CPU 需要的数据在缓存中（cache hit）时，可以快速获取，无需访问较慢的主存。(1分)

(b) 有效访问时间 = 命中率 × 缓存时间 + (1-命中率) × 主存时间
   = 0.95 × 5 + 0.05 × 80
   = 4.75 + 4.0
   = **8.75 ns** (3分)
   
   加速比 = 80 / 8.75 = **9.14 倍**
</details>

### 练习 3：RAID 计算

**题目**：一个服务器有 8 个 4TB 硬盘。

(a) 如果使用 RAID 5，可用容量是多少？(2分)

(b) 如果使用 RAID 6，可用容量是多少？(2分)

(c) 如果使用 RAID 10，可用容量是多少？(2分)

(d) 哪种 RAID 级别最适合数据库服务器？解释原因。(3分)

<details>
<summary>点击查看答案</summary>

(a) RAID 5: (N-1) × 单盘 = (8-1) × 4 = **28 TB** (2分)

(b) RAID 6: (N-2) × 单盘 = (8-2) × 4 = **24 TB** (2分)

(c) RAID 10: N/2 × 单盘 = 8/2 × 4 = **16 TB** (2分)

(d) **RAID 10** 最适合数据库服务器。(1分) 原因：数据库需要高随机读写性能（RAID 10 的条带化提供快速读写），同时需要数据冗余保护（RAID 10 的镜像确保数据安全）。(1分) 虽然 RAID 10 的容量利用率只有 50%，但对于数据库来说，性能和可靠性比存储空间更重要。(1分)
</details>

### 练习 4：SSD 技术

**题目**：

(a) 解释什么是磨损均衡 (Wear Leveling) 以及为什么 SSD 需要它。(3分)

(b) SSD 的 NAND 闪存有哪些类型？它们的主要区别是什么？(4分)

<details>
<summary>点击查看答案</summary>

(a) NAND 闪存的每个存储单元有有限的写入/擦除次数（如 3000-100000 次）。(1分) 磨损均衡是 SSD 控制器的一种算法，确保写入操作均匀分布在所有存储块上，而不是集中在某些"热点"块。(1分) 这样可以防止某些块过早损坏，从而延长整个 SSD 的使用寿命。(1分)

(b) 四种类型：
- SLC (Single-Level Cell): 每单元存1 bit，速度最快，寿命最长，但最贵 (1分)
- MLC (Multi-Level Cell): 每单元存2 bits，速度和寿命适中 (1分)
- TLC (Triple-Level Cell): 每单元存3 bits，更便宜但速度和寿命下降 (1分)
- QLC (Quad-Level Cell): 每单元存4 bits，最便宜但速度最慢、寿命最短 (1分)
</details>

### 练习 5：云存储

**题目**：讨论云存储相对于本地存储的两个优势和两个劣势。(4分)

<details>
<summary>点击查看答案</summary>

**优势**（任选两个，每个1分）：
- 可扩展性：按需增加存储空间，无需购买新硬件
- 随时随地访问：只要有网络就能访问数据
- 自动备份：云服务商维护数据冗余，降低数据丢失风险
- 成本效益：无需维护本地服务器和存储设备

**劣势**（任选两个，每个1分）：
- 依赖网络：没有网络连接就无法访问数据
- 隐私和安全：数据存储在第三方服务器，存在数据泄露风险
- 延迟：访问云端数据比本地存储慢
- 持续成本：长期使用的订阅费用可能超过一次性购买本地存储
</details>

In [ ]:
# ============================================================
# 综合练习：存储性能对比模拟
# ============================================================

def storage_comparison_simulator(file_size_gb=10):
    """模拟不同存储设备读取文件的时间"""
    
    devices = {
        'HDD 5400RPM': 80,     # MB/s
        'HDD 7200RPM': 150,    # MB/s
        'SATA SSD': 550,       # MB/s
        'NVMe SSD': 3500,      # MB/s
        'PCIe 5.0 SSD': 12000, # MB/s
    }
    
    file_size_mb = file_size_gb * 1024
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    names = list(devices.keys())
    speeds = list(devices.values())
    times = [file_size_mb / s for s in speeds]
    
    # 左图：传输速度对比
    colors = ['#95a5a6', '#7f8c8d', '#3498db', '#e74c3c', '#9b59b6']
    ax1.barh(names, speeds, color=colors)
    ax1.set_xlabel('顺序读取速度 (MB/s)', fontsize=11)
    ax1.set_title('存储设备速度对比', fontsize=12, fontweight='bold')
    for i, v in enumerate(speeds):
        ax1.text(v + 100, i, f'{v} MB/s', va='center', fontsize=9, fontweight='bold')
    ax1.set_xlim(0, max(speeds) * 1.3)
    
    # 右图：传输时间
    ax2.barh(names, times, color=colors)
    ax2.set_xlabel(f'读取 {file_size_gb} GB 文件所需时间 (秒)', fontsize=11)
    ax2.set_title(f'读取 {file_size_gb} GB 文件耗时', fontsize=12, fontweight='bold')
    for i, v in enumerate(times):
        if v < 60:
            label = f'{v:.1f} 秒'
        else:
            label = f'{v/60:.1f} 分钟'
        ax2.text(v + 1, i, label, va='center', fontsize=9, fontweight='bold')
    ax2.set_xlim(0, max(times) * 1.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 读取 {file_size_gb} GB 文件的时间对比:")
    for name, speed, t in zip(names, speeds, times):
        time_str = f"{t:.1f}秒" if t < 60 else f"{t/60:.1f}分钟"
        print(f"   {name:>15}: {time_str:>10} ({speed} MB/s)")
    
    speedup = times[0] / times[-1]
    print(f"\n   最快设备比最慢设备快 {speedup:.0f} 倍！")

widgets.interact(
    storage_comparison_simulator,
    file_size_gb=widgets.FloatSlider(min=1, max=100, value=10, step=1, description='文件(GB):')
)

---
## 本笔记本要点回顾

1. **三种 I/O 方式**：程序控制（忙等，最低效）→ 中断驱动（CPU可多任务）→ DMA（CPU几乎不参与，最高效）
2. **存储层次**：寄存器 → L1/L2/L3缓存 → RAM → SSD → HDD → 磁带；越往上速度越快、容量越小、成本越高
3. **缓存**：利用局部性原理，命中率越高性能越好；有效访问时间 = Hit% × CacheTime + Miss% × RAMTime
4. **RAID**：RAID 0(条带/快)、RAID 1(镜像/安全)、RAID 5(条带+校验/平衡)、RAID 6(双校验)、RAID 10(镜像+条带)
5. **SSD vs HDD**：SSD 速度快几千倍（随机IOPS）、无机械部件；HDD 便宜、容量大
6. **SSD 关键技术**：NAND Flash (SLC/MLC/TLC/QLC)、磨损均衡、TRIM、垃圾回收
7. **云存储**：分布式、冗余、可扩展、按需付费；但依赖网络、有隐私风险

> **本章学习完毕！** 请回顾前两个笔记本的内容，完成所有练习题。